# Study 880 — Aggregate Short Interest 🐻

**Is market-wide short interest "arguably the strongest known predictor" of the market?**

Rapach, Ringgenberg & Zhou (2016) build a **market-level** short-interest index and
find it predicts the aggregate equity return with a strong **negative** slope: when
short sellers crowd the whole tape, forward market returns are *lower*. We rebuild the
aggregate index from the **FINRA consolidated short-interest** file (the official
bi-monthly, settlement-date report) for a liquid 50-name panel — the
equal-weight average **days-to-cover** — and run the predictive regression of forward
SPY returns on its detrended level (2017-12-29 → 2026-06-30, 205 bi-monthly
prints).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Availability: aggregate SI is bi-monthly (24/yr) with an ~8-day
publication lag — not a daily series. Index is a days-to-cover average (FINRA has no
shares-outstanding), and the panel is current-membership mega-caps.*


## 1. The idea

Short sellers are, on average, *informed* — so when they pile into the whole market at once (aggregate short interest spikes), it should be a bearish tell for the market as a whole. Rapach-Ringgenberg-Zhou call the detrended aggregate short-interest index the single strongest predictor of the market return, with a *negative* slope: high aggregate SI → lower forward return. We test exactly that regression on a modern, bi-monthly FINRA-built index.

In [1]:
R = dict(h1_beta=-17.2, h1_t=-0.66, h1_r2=0.28, terc_lo=67, terc_hi=21, terc_welch=-0.75)
print('predictive slope (forward SPY return on detrended aggregate SI):')
print('  beta = %+.1f bps per 1sigma of the index   NW t = %+.2f   R2 = %.2f%%'
      % (R['h1_beta'], R['h1_t'], R['h1_r2']))
print('  high-SI periods forward mean %+d bps  vs low-SI %+d bps  (Welch t = %+.2f)'
      % (R['terc_hi'], R['terc_lo'], R['terc_welch']))

predictive slope (forward SPY return on detrended aggregate SI):
  beta = -17.2 bps per 1sigma of the index   NW t = -0.66   R2 = 0.28%
  high-SI periods forward mean +21 bps  vs low-SI +67 bps  (Welch t = -0.75)


## 2. Is the machinery honest? A live synthetic control

We plant the RRZ effect in a seeded toy world (`edge>0`, high detrended SI depresses the next period's return) and check the regression recovers it — and stays *silent* on the null (`edge=0`). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from agg_short import data, strategy as st
null = st.synthetic_detect(data.synthetic_frame(edge=0.0, seed=880, n_periods=200))
planted = st.synthetic_detect(data.synthetic_frame(edge=0.015, seed=880, n_periods=200))
print('null world   : slope NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: slope NW t = %+.2f  (should light up negative)' % planted['t_nw'])

null world   : slope NW t = +0.89  (should be ~0)
planted world: slope NW t = -4.10  (should light up negative)


## 3. The honest verdict — right sign, no significance

On this modern bi-monthly tape the slope has the **correct RRZ sign** (**-17.2 bps** of forward SPY return per 1σ of the index, and it stays negative at every horizon out to 3 months) — but it is **nowhere near significant**: NW *t* = **-0.66**, R² = **0.28%**, and a 5,000-draw permutation places the observed slope only at *p* = 0.22 in the left tail. The high-SI periods do earn a little less (+21 vs +67 bps) but the gap is a coin-flip (Welch *t* = -0.75). The synthetic control recovers a *planted* relation cleanly, so the flat real result is genuine, not a broken engine — the celebrated aggregate-SI predictor simply does **not** show up on a 2017–2026 mega-cap days-to-cover index. **Signal: None** (directionally consistent, statistically absent), **Tradability: Mirage** (the de-risk-on-crowded-shorts overlay just missed the bull market — +8.5%/yr vs +13.2%/yr buy-and-hold).